In [ ]:
import pandas as pd
import numpy as np
import openai
import json
import tiktoken

from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, Filter, FieldCondition, Prefetch, FusionQuery, Distance, PayloadSchemaType, VectorParams, MatchAny

In [ ]:
## retrieve all item ids from amazon items Qdrant Collection (hybrid)

qdrant_client = QdrantClient(url="http://localhost:6333")

dummy_vector = np.zeros(1536).tolist()

In [ ]:
payload = qdrant_client.query_points(
    collection_name = "amazon_items-collection-hybrid-02",
    query = dummy_vector,
    using = "text-embedding-3-small",
    limit = 1000,
    with_payload = ["parent_asin"],
    with_vectors = False,
)

In [ ]:
payload.points

In [ ]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]
parent_asin_list

In [ ]:
df_reviews = pd.read_json("../../data/Electronics_2022_onwards_with_ratings_100_sample_1000.jsonl", lines=True)
df_reviews.head()


In [ ]:
len(df_reviews)

In [ ]:
df_reviews_sample = df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]
df_reviews_sample.head()
df_reviews_sample.shape

### Functions to transform reviews data

In [ ]:
def transform_reviews_data(row):
    """
    Transform the reviews data into a list of dictionaries
    """
    return f"{row["title"]} {row["text"]}"

In [ ]:
encoding = tiktoken.encoding_for_model("text-embedding-3-small")
encoding.encode("I am superman")

In [ ]:
def token_count(row, model="text-embedding-3-small"):
    """
    Count the number of tokens in a string
    """
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(row["transformed_text"]))


In [ ]:
df_reviews_sample["transformed_text"] = df_reviews_sample.apply(transform_reviews_data, axis=1)

In [ ]:
df_reviews_sample["transformed_text_token_count"] = df_reviews_sample.apply(token_count, axis=1)

In [ ]:
df_reviews_sample.head()

In [ ]:
df_reviews_sample = df_reviews_sample[df_reviews_sample["transformed_text_token_count"] < 8192]

In [ ]:
len(df_reviews_sample)

In [ ]:
total_tokens = df_reviews_sample["transformed_text_token_count"].sum()
total_tokens

Create a new Qdrant Collection for reviews

In [ ]:
qdrant_client.create_collection(
    collection_name = "amazon-item-collection-hybrid-01-reviews",
    vectors_config = VectorParams(size=1536, distance=Distance.COSINE),
)

In [ ]:
qdrant_client.create_payload_index(
    collection_name = "amazon-item-collection-hybrid-01-reviews",
    field_name = "parent_asin",
    field_schema = PayloadSchemaType.KEYWORD,
)


In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

In [ ]:
data_to_embed_reviews = df_reviews_sample[["transformed_text", "parent_asin"]].to_dict(orient="records")

In [ ]:
data_to_embed_reviews

In [ ]:
text_to_embed_reviews = [data["transformed_text"] for data in data_to_embed_reviews]

In [ ]:
embeddings_reviews = get_embeddings_batch(text_to_embed_reviews, batch_size=500)
len(embeddings_reviews)

In [ ]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings_reviews, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={
                "text": data["transformed_text"],
                "parent_asin": data["parent_asin"],
            }
        )
    )
    i += 1

In [ ]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="amazon-item-collection-hybrid-01-reviews",
        wait=True,
        points=batch
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

### Retrieval function of user reviews against list of product IDs

In [ ]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-item-collection-hybrid-01-reviews",
        prefetch=[
            Prefetch(
                query=query_embedding,
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    return results

In [ ]:
points = retrieve_prefiltered_reviews_data("cheaper", parent_asins=["B08WQ55H4Y"], k=5)
points

In [ ]:

def retrieve_reviews(query, product_ids, k=5):
    # INSERT_YOUR_CODE
    """
    Retrieve reviews for a given query and list of product IDs.

    Args:
        query (str): The query string to search relevant reviews.
        product_ids (List[str]): A list of product IDs (parent_asin) for which reviews are to be retrieved.
        k (int, optional): The number of top reviews to retrieve. Defaults to 5.

    Returns:
        dict: A dictionary containing:
            - 'retrieved_context_ids': List of product IDs corresponding to each retrieved review.
            - 'retrieved_context': List of review texts retrieved for the query and product IDs.
            - 'similarity_scores': List of similarity scores for each retrieved review.
    """
    qdrant_client = QdrantClient(url="http://localhost:6333")
    
    collection_name = "amazon-item-collection-hybrid-01-reviews"
    k=5
    
    querry_embeddings = get_embedding(query)
    
    response = qdrant_client.query_points(
        collection_name=collection_name,
        prefetch=[Prefetch(
            query=querry_embeddings,
            filter=Filter(
                must=[
                    FieldCondition(key="parent_asin", 
                                    match=MatchAny(any=product_ids))
                    
                    ]
                    
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in response.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["text"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }

def process_reviews_context(context):

    formatted_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, review: {chunk}\n"

    return formatted_context


def get_formatted_reviews_context(query: str, item_list: list, top_k: int = 15) -> str:
    """Get the top k reviews matching a query for a list of prefiltered items.
    
    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered
    
    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    context = retrieve_reviews(query, item_list, top_k)
    formatted_context = process_reviews_context(context)

    return formatted_context

In [ ]:
points = get_formatted_reviews_context("cheaper", item_list=["B08WQ55H4Y"], top_k=5)
points